<a href="https://colab.research.google.com/github/LINWOO0099/Categorical-Encoding/blob/api/logistic_regression_model_predicting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================
# Dataset
# ============================

np.random.seed(42)

n = 200

data = {
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.round(np.random.uniform(20, 120, n), 2),
    'num_services': np.random.randint(1, 8, n),
    'contract_type': np.random.randint(0, 3, n),
}

df = pd.DataFrame(data)


log_odds = (
    -0.05 * df['tenure']
    + 0.03 * df['monthly_charges']
    - 0.2 * df['num_services']
    - 0.8 * df['contract_type']
    + np.random.randn(n) * 0.5
)


prob = 1 / (1 + np.exp(-log_odds))

df['churn'] = (prob > 0.65).astype(int)



# ============================
# Task 1
# Train Test Split + Model
# ============================


X = df.drop('churn', axis=1)

y = df['churn']


X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42

)



# Scale training data only

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)



# Train Logistic Regression

model = LogisticRegression()

model.fit(
    X_train_scaled,
    y_train
)



# Predictions

train_pred = model.predict(X_train_scaled)

test_pred = model.predict(X_test_scaled)



train_acc = accuracy_score(
    y_train,
    train_pred
)

test_acc = accuracy_score(
    y_test,
    test_pred
)


print("Train Accuracy:", train_acc)

print("Test Accuracy:", test_acc)



# Majority class baseline

baseline_accuracy = y_test.value_counts(
    normalize=True
).max()


print(
    "Naive Baseline Accuracy:",
    baseline_accuracy
)





# ============================
# Task 2
# Confusion Matrix + Metrics
# ============================


cm = confusion_matrix(
    y_test,
    test_pred
)


print("\nConfusion Matrix:")

print(cm)



precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)



print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)



TN, FP, FN, TP = cm.ravel()


print("TN:", TN)

print("FP:", FP)

print("FN:", FN)

print("TP:", TP)






# ============================
# Task 3
# Threshold Tuning
# ============================


probabilities = model.predict_proba(
    X_test_scaled
)[:,1]



for threshold in [0.5, 0.3]:

    custom_pred = (
        probabilities >= threshold
    ).astype(int)


    p = precision_score(
        y_test,
        custom_pred,
        zero_division=0
    )


    r = recall_score(
        y_test,
        custom_pred,
        zero_division=0
    )


    print(
        "\nThreshold:",
        threshold
    )

    print(
        "Precision:",
        p
    )

    print(
        "Recall:",
        r
    )





# ============================
# Task 4
# ROC-AUC
# ============================


auc = roc_auc_score(

    y_test,

    probabilities

)


print(
    "\nROC-AUC:",
    auc
)




Train Accuracy: 0.95625
Test Accuracy: 0.9
Naive Baseline Accuracy: 0.85

Confusion Matrix:
[[34  0]
 [ 4  2]]
Precision: 1.0
Recall: 0.3333333333333333
F1 Score: 0.5
TN: 34
FP: 0
FN: 4
TP: 2

Threshold: 0.5
Precision: 1.0
Recall: 0.3333333333333333

Threshold: 0.3
Precision: 1.0
Recall: 1.0

ROC-AUC: 1.0
